# Recurrent neural networks

## 1. The intuition: why does a model need memory?

Notebook 01 ended with two gaps in the embedding lookup:

- It is **static**: "bear" gets the same vector in every sentence.
- It has **no word order**: "The teddy bear" and "bear teddy The" give the same rows.

An RNN is the first fix. It reads the sentence one token at a time and carries a memory along.

> A cute teddy bear is reading.

At each step, the RNN reads one token and updates its **hidden state**, a running summary of everything it has read so far:

![An RNN reading "A cute teddy bear is reading" one token at a time](images/rnn_unrolled.svg)

- $h_t$ is the memory after reading $t$ tokens.
- $h_t$ is computed from the current token $x_t$ and the previous memory $h_{t-1}$.
- The same RNN cell, with the same weights, is reused at every step.

## 2. The mathematics: how does an RNN update its memory?

We keep the sentence "A cute teddy bear is reading". Suppose we have:

- Sequence length $T=6$
- Embedding dimension $d=4$ (as in notebook 01)
- Hidden size $H=3$ (the length of the memory vector)

#### 2.1 The pieces

| Symbol | Name | Intuition | Example |
|---|---|---|---|
| $x_t$ | Input embedding | • The embedding of the token at step $t$<br>• Looked up from $E$, as in notebook 01 | $x_3$ is the row for "teddy": $d = 4$ numbers |
| $h_{t-1}$ | Previous hidden state | • The memory before reading token $t$ | $h_2$ summarizes "A cute" |
| $h_t$ | New hidden state | • The memory after reading token $t$<br>• Same length at every step | $h_3$ summarizes "A cute teddy": $H = 3$ numbers |
| $W_{xh}$ | Input weights | • How the new token enters the memory<br>• Learned, reused at every step | Shape $H \times d = 3 \times 4$ |
| $W_{hh}$ | Memory weights | • How the old memory carries over<br>• Learned, reused at every step | Shape $H \times H = 3 \times 3$ |
| $b$ | Bias | • A learned constant added before tanh<br>• Shifts each memory number up or down | Length $H = 3$ |
| $\tanh$ | Squashing function | • Keeps every number between $-1$ and $1$ | $\tanh(0.80) \approx 0.66$ |

#### 2.2 One step

Every step does the same four things:

1. Multiply the token's embedding by $W_{xh}$, and the old memory by $W_{hh}$.
2. Add both results and a bias $b$.
3. Squash the sum with $\tanh$. The result is the new memory $h_t$.
4. Hand $h_t$ to the next step, where it becomes the old memory.

The weights are learned and shared by every step.

#### 2.3 Step 1 with numbers

Here are small made-up weights, shared by every step, for the first two steps of the sentence:

$$W_{xh} = \begin{bmatrix} 2 & 0 & 2 & 0 \\ 0 & 1 & 0 & 2 \\ 1 & 1 & -1 & 0 \end{bmatrix}, \quad W_{hh} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 1 \\ 1 & 0 & 1 \end{bmatrix}, \quad b = \begin{bmatrix} 0 \\ 0.1 \\ -0.1 \end{bmatrix}$$

**What each matrix does**

- $W_{xh}$ **reads the word.** Row $i$ says which of the word's 4 numbers feed memory number $i$. Row 1, $[2, 0, 2, 0]$, listens to numbers 1 and 3 and ignores 2 and 4.
- $W_{hh}$ **keeps the old memory.** Row $i$ says which of the old 3 memory numbers feed new memory number $i$. Row 1, $[1, 0, 0]$, just keeps old number 1.
- A big weight means "pay attention", a zero means "ignore", and a negative weight pushes the other way (row 3 of $W_{xh}$ subtracts the third number).
- If $W_{hh}$ were all zeros, the memory would forget the past at every step. If $W_{xh}$ were all zeros, it would ignore the words.

All numbers are rounded to 2 decimals at each step.

**How does $W_{xh}$ multiply $x_1$?**

![W_xh (3 by 4) times x_1 (4 by 1) gives 3 numbers](images/rnn_matmul.svg)

- $x_1$ is the embedding of "A": a column of $d = 4$ numbers.
- The inner sizes match: $(3 \times 4)(4 \times 1) = (3 \times 1)$.
- Each row of $W_{xh}$ times $x_1$, added up, gives one number. Row 1 gives $+0.80$.
- $W_{hh} h_{t-1}$ works the same way.

![Step 1: reading "A", starting from a memory of zeros](images/rnn_step1.svg)

- $h_0$ is all zeros, so $W_{hh} h_0$ adds nothing.
- $\tanh$ squashes the sum: $0.80$ becomes $0.66$.
- $h_1$ is the memory after reading "A".

**What does $\tanh$ do?**

- It squashes any number into the range $-1$ to $1$.
- Near 0 it barely changes the number. Far from 0, the curve flattens out near $\pm 1$.
- The two orange points are from step 1: $\tanh(0.80) = 0.66$ and $\tanh(-0.30) = -0.29$.

![The tanh curve, with two numbers from step 1](images/tanh_curve.svg)

#### 2.4 Step 2

![Step 2: reading "cute", starting from h1](images/rnn_step2.svg)

- $h_1$ is now the old memory, so $W_{hh} h_1$ is no longer zero. The memory of "A" flows into step 2.
- $h_2$ depends on both "A" and "cute".
- Step 3 would repeat this with $h_2$ and the embedding of "teddy".

#### 2.5 The update rule

$$\boxed{h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b)}$$

In words: new memory $= f(\text{current token}, \text{old memory})$. The first step starts from $h_0 = 0$.

**Key shapes**

| Piece | Shape | Why |
|---|---|---|
| $W_{xh}\, x_t$ | $(3 \times 4)(4 \times 1) = 3 \times 1$ | Turns a token embedding of length $d$ into length $H$ |
| $W_{hh}\, h_{t-1}$ | $(3 \times 3)(3 \times 1) = 3 \times 1$ | Turns the old memory of length $H$ into length $H$ |
| $h_t$ | $3 \times 1$ | Both products have length $H$, so they can be added |

The memory always has length $H$, whether the RNN has read 6 tokens or 600. A long sequence has to be squeezed into a fixed-size vector.

## 3. Reflection: why do RNNs struggle?

#### 3.1 Slow

![Six RNN steps run one after another, while independent steps could run together](images/rnn_sequential.svg)

- During training, all 6 tokens are known from the start.
- But $h_t$ needs $h_{t-1}$, so step 3 cannot start until step 2 is done.
- The steps run one after another: 6 tokens take 6 time slots, and 600 tokens take 600.
- If the steps did not depend on each other, all of them could run at the same time.

#### 3.2 Forgets

- To learn that "A" mattered, the training signal from the loss has to travel back to step 1.
- On the way, it is multiplied once per step.

![The training signal shrinks as it travels back through the steps](images/rnn_vanishing.svg)

- The factor depends on the weights $W_{hh}$ and the slope of $\tanh$. 0.8 and 1.2 are made-up examples.
- Each factor is below 1 (here 0.8), so the signal shrinks: $0.33$ after 5 steps, $0.01$ after 20.
- The early words get almost no signal, so the RNN struggles to learn from them. This is the **vanishing gradient**.
- If each factor is above 1 (say 1.2), the signal grows instead: $2.5$ after 5 steps, $38$ after 20. This is the **exploding gradient**.
- Where $\tanh$ is flat, near $\pm 1$, its slope is close to 0. That shrinks the signal even more.

#### 3.3 LSTM: a fix, with a limit

- An RNN rewrites its whole memory at every step. An LSTM edits it instead: erase a little, write a little.
- For this, the LSTM (1997) adds a second, long-term memory $c_t$.
- Three gates do the editing. A gate is a number between 0 and 1 for each memory number: 0 blocks, 1 lets through.

![The three LSTM gates, with numbers: forget, input and output](images/lstm_numbers.svg)

- **Forget gate:** how much of each old memory number to keep.
- **Input gate:** how much new information to write.
- **Output gate:** how much of each memory number to show as $h_t$.

**How this helps the vanishing gradient**

- **RNN:** rewrites its memory at every step, like retelling a story. The beginning gets lost.
- **LSTM:** edits its memory, like a notebook: new memory = forget gate $\times$ old memory + input gate $\times$ new information.
- Forget gate 1 and input gate 0: the new note equals the old note. A change at step 1 shows up unchanged at step 20, so the training signal can travel back along it at full strength.
- A gate is rarely exactly 1. With 0.95, 95% of the note survives each step, so $0.36$ of the signal is left after 20 steps, against $0.01$ for the RNN.
- The gates are learned, so the LSTM learns what to keep.

![RNN vs LSTM: how much training signal reaches each step](images/lstm_vs_rnn_signal.svg)

**The limit**

- It is still sequential: step $t$ needs step $t-1$, so it is still slow.
- The memory still has a fixed size.